# COMP9414 Assignment 2: Collaborative Multi-Agent Reinforcement Learning
**Student:** z5726492  
**Term:** T2 2026  
**Due:** Friday, 31 July 2026, 5:00 PM AEST

## Requirements & Imports

In [1]:
!pip install gymnasium numpy matplotlib
!pip install torch torchvision


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from collections import defaultdict
import pickle
import copy
import os

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

---
# Task 1: Environment for Multi-Agent Reinforcement Learning (MARL)

**Deliverables:**
1. Working Python implementation of the grid world environment
2. Visualization of the grid world using Matplotlib
3. Demonstration script with random actions printing state and reward vectors
4. Verification that collision with fragile objects / other robots truncates the episode with correct penalty

### 1.1 Grid World Specification

| Cell Type | Coordinates (row, col) | Amount |
|-----------|----------------------|--------|
| Start Positions | Agent 1: (1,3), Agent 2: (12,3), Agent 3: (7,12) | 3 |
| Fragile Objects | (2,2),(4,4),(6,6),(7,2),(8,4),(10,6),(1,7),(4,7),(8,7),(12,7),(3,11),(8,9),(6,10),(11,11) | 14 |
| Large Dirt | (2,5),(3,9),(4,2),(6,7),(8,8),(9,4),(10,11),(12,2) | 8 |
| Normal Dirt | (1,4),(1,10),(3,2),(4,8),(5,9),(7,3),(9,11),(12,6) | 8 |

In [ ]:
class CleaningGridWorld(gym.Env):
    """12x12 Collaborative Cleaning Grid World for Multi-Agent RL.
    
    N=3 agents cooperatively clean dirt while avoiding fragile objects.
    Observation: per-agent tensor of shape (12, 12, 5) with binary channels:
        0: fragile-cell map
        1: self-position map
        2: other-agent occupancy map
        3: explored-area map
        4: docking-position map (start position of this agent)
    Actions: IDLE=0, UP=1, DOWN=2, LEFT=3, RIGHT=4
    """
    
    IDLE, UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3, 4
    N_AGENTS = 3
    GRID_SIZE = 12
    MAX_STEPS = 50
    
    # Fixed positions (1-indexed as in spec, converted to 0-indexed internally)
    AGENT_STARTS_1IDX = [(1, 3), (12, 3), (7, 12)]
    
    FRAGILE_1IDX = [
        (2, 2), (4, 4), (6, 6), (7, 2), (8, 4), (10, 6),
        (1, 7), (4, 7), (8, 7), (12, 7),
        (3, 11), (8, 9), (6, 10), (11, 11)
    ]
    
    LARGE_DIRT_FIXED_1IDX = [
        (2, 5), (3, 9), (4, 2), (6, 7),
        (8, 8), (9, 4), (10, 11), (12, 2)
    ]
    
    NORMAL_DIRT_FIXED_1IDX = [
        (1, 4), (1, 10), (3, 2), (4, 8),
        (5, 9), (7, 3), (9, 11), (12, 6)
    ]
    
    def __init__(self, mode="fixed", seed=None):
        """
        Args:
            mode: 'fixed' for stationary dirt or 'random' for random dirt placement
            seed: random seed for reproducibility
        """
        super().__init__()
        self.mode = mode
        self.rng = np.random.default_rng(seed)
        
        # Convert 1-indexed to 0-indexed
        self.agent_starts = [(r - 1, c - 1) for r, c in self.AGENT_STARTS_1IDX]
        self.fragile_set = set((r - 1, c - 1) for r, c in self.FRAGILE_1IDX)
        self.large_dirt_fixed = [(r - 1, c - 1) for r, c in self.LARGE_DIRT_FIXED_1IDX]
        self.normal_dirt_fixed = [(r - 1, c - 1) for r, c in self.NORMAL_DIRT_FIXED_1IDX]
        
        # Action and observation spaces
        self.action_space = spaces.MultiDiscrete([5] * self.N_AGENTS)
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(self.N_AGENTS, self.GRID_SIZE, self.GRID_SIZE, 5),
            dtype=np.float32
        )
        
        # Action deltas: IDLE, UP, DOWN, LEFT, RIGHT
        self.action_deltas = {
            0: (0, 0),   # IDLE
            1: (-1, 0),  # UP (row - 1)
            2: (1, 0),   # DOWN (row + 1)
            3: (0, -1),  # LEFT (col - 1)
            4: (0, 1),   # RIGHT (col + 1)
        }
    
    def _place_dirt(self):
        """Place dirt on the grid. Fixed mode uses spec positions; random mode randomizes."""
        self.large_dirt = {}   # (row, col) -> True
        self.normal_dirt = {}  # (row, col) -> True
        
        if self.mode == "fixed":
            for pos in self.large_dirt_fixed:
                self.large_dirt[pos] = True
            for pos in self.normal_dirt_fixed:
                self.normal_dirt[pos] = True
        else:
            # Random placement: same counts, avoid fragile and start positions
            occupied = self.fragile_set | set(self.agent_starts)
            all_cells = [
                (r, c) for r in range(self.GRID_SIZE) for c in range(self.GRID_SIZE)
                if (r, c) not in occupied
            ]
            chosen = self.rng.choice(len(all_cells), size=16, replace=False)
            for i in range(8):
                self.large_dirt[all_cells[chosen[i]]] = True
            for i in range(8, 16):
                self.normal_dirt[all_cells[chosen[i]]] = True
    
    def reset(self, seed=None, options=None):
        """Reset the environment to initial state."""
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        
        self.step_count = 0
        self.agent_positions = [list(pos) for pos in self.agent_starts]
        
        # Place dirt
        self._place_dirt()
        self.total_dirt_value = len(self.large_dirt) * 2 + len(self.normal_dirt)  # large=2x weight
        self.initial_dirt_value = self.total_dirt_value
        
        # Explored cells per agent: agent_id -> set of (row, col)
        self.explored = [set() for _ in range(self.N_AGENTS)]
        for i, pos in enumerate(self.agent_positions):
            self.explored[i].add(tuple(pos))
        
        # Global explored map: who first explored each cell
        self.cell_owner = {}  # (row, col) -> agent_id
        for i, pos in enumerate(self.agent_positions):
            self.cell_owner[tuple(pos)] = i
        
        # Tracking for metrics
        self.visit_map = np.zeros((self.GRID_SIZE, self.GRID_SIZE), dtype=int)
        self.revisit_map = np.zeros((self.GRID_SIZE, self.GRID_SIZE), dtype=int)
        for pos in self.agent_positions:
            self.visit_map[pos[0], pos[1]] += 1
        
        observations = self._get_observations()
        info = self._get_info()
        return observations, info
    
    def _get_observations(self):
        """Build per-agent observation tensors of shape (12, 12, 5)."""
        obs = np.zeros((self.N_AGENTS, self.GRID_SIZE, self.GRID_SIZE, 5), dtype=np.float32)
        
        for i in range(self.N_AGENTS):
            # Channel 0: Fragile cell map (same for all agents)
            for (r, c) in self.fragile_set:
                obs[i, r, c, 0] = 1.0
            
            # Channel 1: Self-position
            r, c = self.agent_positions[i]
            obs[i, r, c, 1] = 1.0
            
            # Channel 2: Other agents' positions
            for j in range(self.N_AGENTS):
                if j != i:
                    r2, c2 = self.agent_positions[j]
                    obs[i, r2, c2, 2] = 1.0
            
            # Channel 3: Explored area map (union of all explored cells)
            for cell in self.cell_owner:
                obs[i, cell[0], cell[1], 3] = 1.0
            
            # Channel 4: Docking position (this agent's start)
            sr, sc = self.agent_starts[i]
            obs[i, sr, sc, 4] = 1.0
        
        return obs
    
    def _get_global_state(self):
        """Build global state by stacking all agent observations. Used by MAPPO critic."""
        obs = self._get_observations()
        # Concatenate along channel dimension: (12, 12, 5*N_AGENTS)
        return np.concatenate([obs[i] for i in range(self.N_AGENTS)], axis=-1)
    
    def _get_info(self):
        """Return info dict with current environment state."""
        remaining_large = len(self.large_dirt)
        remaining_normal = len(self.normal_dirt)
        return {
            "step": self.step_count,
            "remaining_large_dirt": remaining_large,
            "remaining_normal_dirt": remaining_normal,
            "remaining_dirt_value": remaining_large * 2 + remaining_normal,
            "agent_positions": [tuple(p) for p in self.agent_positions],
        }
    
    def step(self, actions):
        """Execute joint actions for all agents.
        
        Args:
            actions: list/array of N_AGENTS actions, each in {0,1,2,3,4}
            
        Returns:
            observations, rewards, terminated, truncated, info
        """
        assert len(actions) == self.N_AGENTS
        self.step_count += 1
        
        rewards = np.zeros(self.N_AGENTS, dtype=np.float32)
        terminated = False
        truncated = False
        
        # Compute proposed new positions
        proposed = []
        for i in range(self.N_AGENTS):
            dr, dc = self.action_deltas[actions[i]]
            new_r = self.agent_positions[i][0] + dr
            new_c = self.agent_positions[i][1] + dc
            proposed.append((new_r, new_c))
        
        # Safety shield: check for collisions and out-of-bounds
        for i in range(self.N_AGENTS):
            new_r, new_c = proposed[i]
            
            # Out of bounds check
            if new_r < 0 or new_r >= self.GRID_SIZE or new_c < 0 or new_c >= self.GRID_SIZE:
                rewards[i] += -20.0
                truncated = True
                # Force IDLE via safety shield
                proposed[i] = tuple(self.agent_positions[i])
                actions[i] = self.IDLE
                continue
            
            # Fragile object collision
            if (new_r, new_c) in self.fragile_set:
                rewards[i] += -20.0
                truncated = True
                proposed[i] = tuple(self.agent_positions[i])
                actions[i] = self.IDLE
                continue
        
        # Check agent-agent collisions (two agents moving to same cell)
        for i in range(self.N_AGENTS):
            for j in range(i + 1, self.N_AGENTS):
                if proposed[i] == proposed[j] and proposed[i] != tuple(self.agent_positions[i]):
                    rewards[i] += -20.0
                    rewards[j] += -20.0
                    truncated = True
                    proposed[i] = tuple(self.agent_positions[i])
                    proposed[j] = tuple(self.agent_positions[j])
                    actions[i] = self.IDLE
                    actions[j] = self.IDLE
        
        # Apply moves and compute rewards
        for i in range(self.N_AGENTS):
            new_pos = proposed[i]
            self.agent_positions[i] = list(new_pos)
            
            # Track visits
            self.visit_map[new_pos[0], new_pos[1]] += 1
            
            # Idle penalty
            if actions[i] == self.IDLE:
                rewards[i] += -0.1
            
            # Exploration reward / revisit penalty
            if new_pos not in self.cell_owner:
                # New cell explored
                self.cell_owner[new_pos] = i
                self.explored[i].add(new_pos)
                rewards[i] += 2.0  # exploration reward
            elif actions[i] != self.IDLE:
                # Revisiting an already-explored cell
                self.revisit_map[new_pos[0], new_pos[1]] += 1
                rewards[i] += -1.0  # revisit penalty
            
            # Dirt cleaning
            if new_pos in self.large_dirt:
                del self.large_dirt[new_pos]
                rewards[i] += 4.0
            elif new_pos in self.normal_dirt:
                del self.normal_dirt[new_pos]
                rewards[i] += 2.0
        
        # Timestep penalty (shared across team)
        for i in range(self.N_AGENTS):
            rewards[i] += -0.1
        
        # Check completion
        if len(self.large_dirt) == 0 and len(self.normal_dirt) == 0:
            for i in range(self.N_AGENTS):
                rewards[i] += 20.0
            terminated = True
        
        # Check time limit
        if self.step_count >= self.MAX_STEPS and not terminated:
            truncated = True
        
        observations = self._get_observations()
        info = self._get_info()
        
        return observations, rewards, terminated, truncated, info
    
    def get_success_rate(self):
        """Compute success rate: percentage of dirt value cleaned."""
        remaining = len(self.large_dirt) * 2 + len(self.normal_dirt)
        return 1.0 - (remaining / self.initial_dirt_value) if self.initial_dirt_value > 0 else 1.0
    
    def get_global_rcr(self):
        """Compute Global Redundant Coverage Ratio."""
        total_visits = self.visit_map.sum()
        total_revisits = self.revisit_map.sum()
        return total_revisits / max(1, total_visits)
    
    def render(self, ax=None):
        """Visualize the grid world using Matplotlib."""
        if ax is None:
            fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        
        grid = np.ones((self.GRID_SIZE, self.GRID_SIZE, 3))  # white background
        
        # Color explored cells light gray
        agent_colors_light = [
            [1.0, 1.0, 0.7],  # light yellow (agent 0)
            [0.7, 1.0, 0.7],  # light green (agent 1)
            [0.7, 0.7, 1.0],  # light blue (agent 2)
        ]
        for cell, owner in self.cell_owner.items():
            grid[cell[0], cell[1]] = agent_colors_light[owner]
        
        # Fragile objects (red)
        for r, c in self.fragile_set:
            grid[r, c] = [1.0, 0.4, 0.4]
        
        # Large dirt (dark gray)
        for r, c in self.large_dirt:
            grid[r, c] = [0.4, 0.4, 0.4]
        
        # Normal dirt (light gray)
        for r, c in self.normal_dirt:
            grid[r, c] = [0.7, 0.7, 0.7]
        
        # Agents
        agent_colors = [
            [1.0, 1.0, 0.0],  # yellow (agent 0)
            [0.0, 0.8, 0.0],  # green (agent 1)
            [0.0, 0.4, 1.0],  # blue (agent 2)
        ]
        for i, pos in enumerate(self.agent_positions):
            grid[pos[0], pos[1]] = agent_colors[i]
        
        ax.imshow(grid, origin='upper')
        ax.set_xticks(range(self.GRID_SIZE))
        ax.set_yticks(range(self.GRID_SIZE))
        ax.set_xticklabels(range(1, self.GRID_SIZE + 1))
        ax.set_yticklabels(range(1, self.GRID_SIZE + 1))
        ax.grid(True, linewidth=0.5, color='black')
        ax.set_title(f"Step {self.step_count} | Dirt remaining: L={len(self.large_dirt)} N={len(self.normal_dirt)}")
        
        # Legend
        legend_items = [
            mpatches.Patch(color=[1, 0.4, 0.4], label='Fragile'),
            mpatches.Patch(color=[0.4, 0.4, 0.4], label='Large Dirt'),
            mpatches.Patch(color=[0.7, 0.7, 0.7], label='Normal Dirt'),
            mpatches.Patch(color=[1, 1, 0], label='Agent 1'),
            mpatches.Patch(color=[0, 0.8, 0], label='Agent 2'),
            mpatches.Patch(color=[0, 0.4, 1], label='Agent 3'),
        ]
        ax.legend(handles=legend_items, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
        
        return ax

### 1.2 Visualize the Grid World

In [ ]:
env = CleaningGridWorld(mode="fixed")
obs, info = env.reset()

fig, ax = plt.subplots(1, 1, figsize=(9, 8))
env.render(ax=ax)
plt.tight_layout()
plt.show()

print(f"Observation shape per agent: {obs[0].shape}")
print(f"Info: {info}")

### 1.3 Random Action Demonstration

In [ ]:
env = CleaningGridWorld(mode="fixed")
obs, info = env.reset(seed=42)

action_names = {0: 'IDLE', 1: 'UP', 2: 'DOWN', 3: 'LEFT', 4: 'RIGHT'}

print("=== Random Action Demo ===")
print(f"Initial positions: {info['agent_positions']}")
print(f"Initial dirt: Large={info['remaining_large_dirt']}, Normal={info['remaining_normal_dirt']}")
print()

for step in range(10):
    actions = [np.random.randint(0, 5) for _ in range(env.N_AGENTS)]
    obs, rewards, terminated, truncated, info = env.step(actions)
    
    action_str = [action_names[a] for a in actions]
    print(f"Step {step + 1}: Actions={action_str}")
    print(f"  Positions: {info['agent_positions']}")
    print(f"  Rewards: {rewards}")
    print(f"  Dirt remaining: L={info['remaining_large_dirt']}, N={info['remaining_normal_dirt']}")
    
    if terminated:
        print("  >> TERMINATED (all dirt cleaned!)")
        break
    if truncated:
        print("  >> TRUNCATED (collision or time limit)")
        break
    print()

### 1.4 Collision Verification

In [ ]:
def verify_collision_penalties():
    """Verify that collisions with fragile objects and OOB correctly truncate with -20 penalty."""
    
    # Test 1: Out of bounds
    print("=== Test 1: Out of Bounds ===")
    env = CleaningGridWorld(mode="fixed")
    env.reset()
    # Agent 1 starts at (0, 2) in 0-indexed. Moving UP goes to (-1, 2) -> OOB
    actions = [env.UP, env.IDLE, env.IDLE]
    obs, rewards, terminated, truncated, info = env.step(actions)
    print(f"  Agent 1 moved UP from row 0 -> Reward: {rewards[0]}, Truncated: {truncated}")
    assert truncated, "Should be truncated on OOB"
    assert rewards[0] <= -20.0, f"Expected -20 penalty, got {rewards[0]}"
    print("  PASSED!")
    
    # Test 2: Fragile object collision
    print("\n=== Test 2: Fragile Object Collision ===")
    env = CleaningGridWorld(mode="fixed")
    env.reset()
    # Agent 1 starts at (0, 2). Fragile at (1, 1) in 0-indexed.
    # Move agent 1 DOWN to (1, 2), then LEFT to (1, 1) which is fragile
    env.step([env.DOWN, env.IDLE, env.IDLE])  # move to (1, 2)
    obs, rewards, terminated, truncated, info = env.step([env.LEFT, env.IDLE, env.IDLE])
    print(f"  Agent 1 moved into fragile cell -> Reward: {rewards[0]}, Truncated: {truncated}")
    assert truncated, "Should be truncated on fragile collision"
    assert rewards[0] <= -20.0, f"Expected -20 penalty, got {rewards[0]}"
    print("  PASSED!")
    
    # Test 3: Agent-agent collision
    print("\n=== Test 3: Agent-Agent Collision ===")
    env = CleaningGridWorld(mode="fixed")
    env.reset()
    # We'll manually place agents close together for testing
    env.agent_positions = [[5, 5], [5, 7], [0, 0]]
    # Both agents 0 and 1 move toward (5, 6)
    obs, rewards, terminated, truncated, info = env.step([env.RIGHT, env.LEFT, env.IDLE])
    print(f"  Agents 1&2 collide at same cell -> Rewards: {rewards[:2]}, Truncated: {truncated}")
    assert truncated, "Should be truncated on agent collision"
    assert rewards[0] <= -20.0 and rewards[1] <= -20.0, "Both agents should get -20"
    print("  PASSED!")
    
    print("\nAll collision tests passed!")

verify_collision_penalties()

---
# Task 2: Modular PPO Pipeline Construction

**Deliverables:**
1. Actor and Critic networks with sanity check
2. Rollout buffer module with test
3. Advantage estimator (GAE) module with test
4. Optimizer module with test

### 2.1 Actor and Critic Networks (Algorithm 1)

In [ ]:
class ActorNetwork(nn.Module):
    """Policy network: maps observation (12,12,5) -> action probabilities over 5 actions.
    
    Uses a CNN to process the spatial observation, followed by MLP layers.
    For MAPPO shared actor, an additional one-hot agent ID is concatenated.
    """
    
    def __init__(self, obs_channels=5, n_actions=5, hidden_sizes=(128, 64), use_agent_id=False, n_agents=3):
        super().__init__()
        self.use_agent_id = use_agent_id
        self.n_agents = n_agents
        
        # Flatten spatial observation
        input_size = 12 * 12 * obs_channels
        if use_agent_id:
            input_size += n_agents  # one-hot agent ID
        
        layers = []
        prev_size = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_size, h))
            layers.append(nn.ReLU())
            prev_size = h
        layers.append(nn.Linear(prev_size, n_actions))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, obs, agent_id=None):
        """Forward pass.
        
        Args:
            obs: tensor of shape (batch, 12, 12, 5)
            agent_id: int, required if use_agent_id=True
        Returns:
            action_probs: tensor of shape (batch, 5)
        """
        x = obs.reshape(obs.shape[0], -1)  # flatten
        
        if self.use_agent_id and agent_id is not None:
            one_hot = torch.zeros(obs.shape[0], self.n_agents, device=obs.device)
            one_hot[:, agent_id] = 1.0
            x = torch.cat([x, one_hot], dim=-1)
        
        logits = self.network(x)
        return F.softmax(logits, dim=-1)


class CriticNetwork(nn.Module):
    """Value network: maps observation -> scalar state value.
    
    IPPO critic: input is local obs (12,12,5)
    MAPPO critic: input is global state (12,12,15) = concatenation of all agents' obs
    """
    
    def __init__(self, obs_channels=5, hidden_sizes=(128, 64)):
        super().__init__()
        input_size = 12 * 12 * obs_channels
        
        layers = []
        prev_size = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_size, h))
            layers.append(nn.ReLU())
            prev_size = h
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, obs):
        """Forward pass.
        
        Args:
            obs: tensor of shape (batch, 12, 12, C)
        Returns:
            value: tensor of shape (batch, 1)
        """
        x = obs.reshape(obs.shape[0], -1)
        return self.network(x)

In [ ]:
# Sanity check: Actor and Critic networks
print("=== Actor Network (IPPO) ===")
actor = ActorNetwork(obs_channels=5, hidden_sizes=(128, 64))
print(actor)
dummy_obs = torch.randn(4, 12, 12, 5)  # batch of 4
probs = actor(dummy_obs)
print(f"Input shape: {dummy_obs.shape} -> Output shape: {probs.shape}")
print(f"Probabilities sum to 1: {probs.sum(dim=-1)}")

print("\n=== Actor Network (MAPPO - shared with agent ID) ===")
actor_mappo = ActorNetwork(obs_channels=5, hidden_sizes=(128, 64), use_agent_id=True, n_agents=3)
probs_mappo = actor_mappo(dummy_obs, agent_id=0)
print(f"Output shape with agent_id: {probs_mappo.shape}")

print("\n=== Critic Network (IPPO - local obs) ===")
critic_ippo = CriticNetwork(obs_channels=5, hidden_sizes=(128, 64))
print(critic_ippo)
value = critic_ippo(dummy_obs)
print(f"Input shape: {dummy_obs.shape} -> Value shape: {value.shape}")

print("\n=== Critic Network (MAPPO - global state) ===")
critic_mappo = CriticNetwork(obs_channels=15, hidden_sizes=(128, 64))  # 5 channels * 3 agents
dummy_global = torch.randn(4, 12, 12, 15)
value_mappo = critic_mappo(dummy_global)
print(f"Input shape: {dummy_global.shape} -> Value shape: {value_mappo.shape}")

print("\nAll network sanity checks passed!")

### 2.2 Rollout Buffer Module (Algorithm 2)

In [ ]:
class RolloutBuffer:
    """Stores trajectory data for one agent during a rollout.
    
    Stores: observations, actions, rewards, log-probabilities, values.
    After advantage estimation, also stores: advantages, returns.
    """
    
    def __init__(self):
        self.observations = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.advantages = []
        self.returns = []
        # For MAPPO: also store global states
        self.global_states = []
    
    def store(self, obs, action, reward, log_prob, value, global_state=None):
        """Store a single transition."""
        self.observations.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)
        self.log_probs.append(log_prob)
        self.values.append(value)
        if global_state is not None:
            self.global_states.append(global_state)
    
    def to_tensors(self):
        """Convert stored lists to tensors for training."""
        data = {
            'observations': torch.FloatTensor(np.array(self.observations)),
            'actions': torch.LongTensor(np.array(self.actions)),
            'rewards': torch.FloatTensor(np.array(self.rewards)),
            'log_probs': torch.FloatTensor(np.array(self.log_probs)),
            'values': torch.FloatTensor(np.array(self.values)),
        }
        if self.advantages:
            data['advantages'] = torch.FloatTensor(np.array(self.advantages))
            data['returns'] = torch.FloatTensor(np.array(self.returns))
        if self.global_states:
            data['global_states'] = torch.FloatTensor(np.array(self.global_states))
        return data
    
    def clear(self):
        """Clear all stored data."""
        self.__init__()
    
    def __len__(self):
        return len(self.observations)


def collect_rollout(env, actors, critics, n_steps, use_global_state=False):
    """Collect rollout data from the environment using current policies.
    
    Implements Algorithm 2 (IPPO) / Algorithm 6 (MAPPO).
    
    Args:
        env: CleaningGridWorld environment
        actors: list of ActorNetwork (one per agent for IPPO, or single shared for MAPPO)
        critics: list of CriticNetwork (one per agent)
        n_steps: number of environment steps to collect
        use_global_state: if True, store global state for MAPPO critic
        
    Returns:
        buffers: list of RolloutBuffer, one per agent
    """
    n_agents = env.N_AGENTS
    buffers = [RolloutBuffer() for _ in range(n_agents)]
    
    obs, info = env.reset()
    
    for t in range(n_steps):
        actions = []
        
        global_state = None
        if use_global_state:
            global_state = env._get_global_state()
        
        for i in range(n_agents):
            obs_tensor = torch.FloatTensor(obs[i]).unsqueeze(0).to(device)
            
            with torch.no_grad():
                # Get action probabilities
                if hasattr(actors[i], 'use_agent_id') and actors[i].use_agent_id:
                    action_probs = actors[i](obs_tensor, agent_id=i)
                else:
                    action_probs = actors[i](obs_tensor)
                
                dist = Categorical(action_probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                
                # Get value estimate
                if use_global_state:
                    gs_tensor = torch.FloatTensor(global_state).unsqueeze(0).to(device)
                    value = critics[i](gs_tensor)
                else:
                    value = critics[i](obs_tensor)
            
            actions.append(action.item())
            buffers[i].store(
                obs=obs[i],
                action=action.item(),
                reward=0.0,  # filled after env.step
                log_prob=log_prob.item(),
                value=value.item(),
                global_state=global_state if use_global_state else None
            )
        
        # Step environment
        obs, rewards, terminated, truncated, info = env.step(actions)
        
        # Update rewards in buffers
        for i in range(n_agents):
            buffers[i].rewards[-1] = rewards[i]
        
        if terminated or truncated:
            obs, info = env.reset()
    
    return buffers

In [ ]:
# Test: Rollout buffer module
print("=== Rollout Buffer Test ===")
env_test = CleaningGridWorld(mode="fixed")
env_test.reset(seed=42)

test_actors = [ActorNetwork().to(device) for _ in range(3)]
test_critics = [CriticNetwork().to(device) for _ in range(3)]

buffers = collect_rollout(env_test, test_actors, test_critics, n_steps=50)

for i, buf in enumerate(buffers):
    data = buf.to_tensors()
    print(f"Agent {i}: {len(buf)} transitions")
    print(f"  obs shape: {data['observations'].shape}")
    print(f"  actions shape: {data['actions'].shape}")
    print(f"  rewards shape: {data['rewards'].shape}")
    print(f"  log_probs shape: {data['log_probs'].shape}")
    print(f"  values shape: {data['values'].shape}")

# Save test data
test_data = {f'agent_{i}': buffers[i].to_tensors() for i in range(3)}
np.savez('rollout_test_data.npz', 
         **{f'{k}_{dk}': v.numpy() for k, d in test_data.items() for dk, v in d.items()})
print("\nRollout test data saved to rollout_test_data.npz")
print("Rollout buffer test passed!")

### 2.3 Advantage Estimator Module - GAE (Algorithm 3)

In [ ]:
def compute_gae(rewards, values, gamma=0.95, lam=0.95):
    """Compute Generalized Advantage Estimation (GAE).
    
    Implements Algorithm 3 (IPPO) / Algorithm 7 (MAPPO).
    
    Args:
        rewards: array of shape (T,) - rewards at each timestep
        values: array of shape (T,) - value estimates at each timestep
        gamma: discount factor
        lam: GAE smoothing parameter
        
    Returns:
        advantages: array of shape (T,)
        returns: array of shape (T,) - advantage + value (target for critic)
    """
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    
    # Initialize A_t = 0 for terminal step
    gae = 0.0
    
    for t in reversed(range(T)):
        if t == T - 1:
            next_value = 0.0  # terminal
        else:
            next_value = values[t + 1]
        
        # TD error: delta = r_t + gamma * V(s_{t+1}) - V(s_t)
        delta = rewards[t] + gamma * next_value - values[t]
        
        # GAE: A_t = delta_t + gamma * lambda * A_{t+1}
        gae = delta + gamma * lam * gae
        advantages[t] = gae
    
    # Return target: R_t = A_t + V(s_t)
    returns = advantages + values
    
    return advantages, returns


def compute_advantages_for_buffer(buffer, gamma=0.95, lam=0.95):
    """Compute and store GAE advantages in a RolloutBuffer."""
    rewards = np.array(buffer.rewards)
    values = np.array(buffer.values)
    
    advantages, returns = compute_gae(rewards, values, gamma, lam)
    
    buffer.advantages = advantages.tolist()
    buffer.returns = returns.tolist()
    
    return advantages, returns

In [ ]:
# Test: Advantage estimator module
print("=== GAE Advantage Estimator Test ===")

# Use rollout data from previous test
for i, buf in enumerate(buffers):
    advantages, returns = compute_advantages_for_buffer(buf, gamma=0.95, lam=0.95)
    print(f"Agent {i}:")
    print(f"  Advantages - mean: {advantages.mean():.4f}, std: {advantages.std():.4f}")
    print(f"  Returns - mean: {returns.mean():.4f}, std: {returns.std():.4f}")
    print(f"  Advantage range: [{advantages.min():.4f}, {advantages.max():.4f}]")

# Bias/variance analysis
print("\n=== Bias & Variance Analysis ===")
for gamma_test in [0.90, 0.95, 0.98]:
    for lam_test in [0.90, 0.95, 0.98]:
        adv, _ = compute_gae(
            np.array(buffers[0].rewards),
            np.array(buffers[0].values),
            gamma=gamma_test, lam=lam_test
        )
        print(f"  gamma={gamma_test}, lambda={lam_test}: mean={adv.mean():.4f}, var={adv.var():.4f}")

print("\nAdvantage estimator test passed!")

### 2.4 PPO Optimizer Module (Algorithm 4)

In [ ]:
class PPOOptimizer:
    """PPO optimization module implementing Algorithm 4 (IPPO) / Algorithm 8 (MAPPO).
    
    Computes clipped surrogate actor loss, critic value loss, and entropy bonus.
    """
    
    def __init__(self, actor, critic, actor_lr=3e-4, critic_lr=3e-4,
                 epsilon=0.2, c1=0.5, c2=0.01, epochs=10, minibatch_size=200,
                 weight_decay=1e-5, use_agent_id=False, agent_id=None):
        self.actor = actor
        self.critic = critic
        self.epsilon = epsilon
        self.c1 = c1  # critic loss coefficient
        self.c2 = c2  # entropy bonus coefficient
        self.epochs = epochs
        self.minibatch_size = minibatch_size
        self.use_agent_id = use_agent_id
        self.agent_id = agent_id
        
        self.actor_optimizer = optim.Adam(actor.parameters(), lr=actor_lr, weight_decay=weight_decay)
        self.critic_optimizer = optim.Adam(critic.parameters(), lr=critic_lr, weight_decay=weight_decay)
    
    def compute_losses(self, obs, actions, old_log_probs, advantages, returns,
                       global_states=None):
        """Compute actor (clipped), critic, and entropy losses for a minibatch.
        
        Returns:
            actor_loss, critic_loss, entropy, total_loss, prob_ratio
        """
        # New action probabilities
        if self.use_agent_id and self.agent_id is not None:
            action_probs = self.actor(obs, agent_id=self.agent_id)
        else:
            action_probs = self.actor(obs)
        
        dist = Categorical(action_probs)
        new_log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        
        # Probability ratio
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        # Clipped surrogate actor loss
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        
        # Critic value loss
        if global_states is not None:
            values = self.critic(global_states).squeeze(-1)
        else:
            values = self.critic(obs).squeeze(-1)
        critic_loss = F.mse_loss(values, returns)
        
        # Entropy bonus
        entropy_bonus = entropy.mean()
        
        # Total loss
        total_loss = actor_loss + self.c1 * critic_loss - self.c2 * entropy_bonus
        
        return actor_loss, critic_loss, entropy_bonus, total_loss, ratio.mean()
    
    def update(self, buffer_data):
        """Perform multiple epochs of PPO updates on buffer data.
        
        Args:
            buffer_data: dict from RolloutBuffer.to_tensors()
            
        Returns:
            metrics: dict of average losses over all epochs
        """
        obs = buffer_data['observations'].to(device)
        actions = buffer_data['actions'].to(device)
        old_log_probs = buffer_data['log_probs'].to(device)
        advantages = buffer_data['advantages'].to(device)
        returns = buffer_data['returns'].to(device)
        global_states = buffer_data.get('global_states')
        if global_states is not None:
            global_states = global_states.to(device)
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        n_samples = len(obs)
        metrics = defaultdict(list)
        
        for epoch in range(self.epochs):
            # Shuffle and create minibatches
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            
            for start in range(0, n_samples, self.minibatch_size):
                end = min(start + self.minibatch_size, n_samples)
                mb_idx = indices[start:end]
                
                mb_obs = obs[mb_idx]
                mb_actions = actions[mb_idx]
                mb_old_log_probs = old_log_probs[mb_idx]
                mb_advantages = advantages[mb_idx]
                mb_returns = returns[mb_idx]
                mb_gs = global_states[mb_idx] if global_states is not None else None
                
                actor_loss, critic_loss, entropy, total_loss, ratio = self.compute_losses(
                    mb_obs, mb_actions, mb_old_log_probs, mb_advantages, mb_returns, mb_gs
                )
                
                # Update networks
                self.actor_optimizer.zero_grad()
                self.critic_optimizer.zero_grad()
                total_loss.backward()
                self.actor_optimizer.step()
                self.critic_optimizer.step()
                
                metrics['actor_loss'].append(actor_loss.item())
                metrics['critic_loss'].append(critic_loss.item())
                metrics['entropy'].append(entropy.item())
                metrics['total_loss'].append(total_loss.item())
                metrics['ratio'].append(ratio.item())
        
        return {k: np.mean(v) for k, v in metrics.items()}

In [ ]:
# Test: Optimizer module
print("=== PPO Optimizer Test ===")

# Create test networks and optimizer
test_actor = ActorNetwork(obs_channels=5, hidden_sizes=(64, 32)).to(device)
test_critic = CriticNetwork(obs_channels=5, hidden_sizes=(64, 32)).to(device)

optimizer = PPOOptimizer(
    test_actor, test_critic,
    actor_lr=3e-4, critic_lr=3e-4,
    epsilon=0.2, epochs=4, minibatch_size=25
)

# Use agent 0's buffer data
compute_advantages_for_buffer(buffers[0], gamma=0.95, lam=0.95)
test_data = buffers[0].to_tensors()

# Single batch loss computation
obs_t = test_data['observations'].to(device)
act_t = test_data['actions'].to(device)
lp_t = test_data['log_probs'].to(device)
adv_t = test_data['advantages'].to(device)
ret_t = test_data['returns'].to(device)

actor_loss, critic_loss, entropy, total_loss, ratio = optimizer.compute_losses(
    obs_t, act_t, lp_t, adv_t, ret_t
)
print(f"Actor loss: {actor_loss.item():.4f}")
print(f"Critic loss: {critic_loss.item():.4f}")
print(f"Entropy bonus: {entropy.item():.4f}")
print(f"Total loss: {total_loss.item():.4f}")
print(f"Probability ratio: {ratio.item():.4f}")

# Full update
print("\nRunning PPO update (4 epochs)...")
metrics = optimizer.update(test_data)
print(f"Average metrics: {metrics}")

print("\nOptimizer test passed!")

---
# Task 3: Independent Proximal Policy Optimization (IPPO)

**Deliverables:**
1. Modular codebase showing step-by-step sub-task fulfillment
2. Training results: training curves (total loss, actor loss, critic loss, entropy) for fixed & random modes
3. Performance evaluation: episode return, episode length, success rate, safety violations
4. Visual demonstration: load best checkpoint and run in simulation

### 3.1 IPPO Hyperparameters

In [ ]:
IPPO_CONFIG = {
    'gamma': 0.95,
    'lam': 0.95,
    'epsilon': 0.2,
    'actor_lr': 5e-4,
    'critic_lr': 5e-4,
    'rollout_length': 400,
    'minibatch_size': 200,
    'epochs': 10,
    'max_iterations': 500,
    'hidden_sizes': (128, 64),
    'c1': 0.5,         # critic loss coefficient
    'c2': 0.01,        # entropy bonus coefficient
    'weight_decay': 1e-5,
    'seed': 42,
}

print("IPPO Hyperparameters:")
for k, v in IPPO_CONFIG.items():
    print(f"  {k}: {v}")

### 3.2 IPPO Training Loop

In [ ]:
def train_ippo(config, mode="fixed", verbose=True):
    """Train IPPO: independent actor-critic per agent.
    
    Implements Algorithms 1-4.
    """
    env = CleaningGridWorld(mode=mode, seed=config['seed'])
    n_agents = env.N_AGENTS
    
    # Phase 1A/1B: Initialize independent actor and critic for each agent
    actors = [ActorNetwork(obs_channels=5, hidden_sizes=config['hidden_sizes']).to(device)
              for _ in range(n_agents)]
    critics = [CriticNetwork(obs_channels=5, hidden_sizes=config['hidden_sizes']).to(device)
               for _ in range(n_agents)]
    
    optimizers = [
        PPOOptimizer(
            actors[i], critics[i],
            actor_lr=config['actor_lr'],
            critic_lr=config['critic_lr'],
            epsilon=config['epsilon'],
            c1=config['c1'],
            c2=config['c2'],
            epochs=config['epochs'],
            minibatch_size=config['minibatch_size'],
            weight_decay=config['weight_decay'],
        )
        for i in range(n_agents)
    ]
    
    # Training history
    history = {
        'total_loss': [[] for _ in range(n_agents)],
        'actor_loss': [[] for _ in range(n_agents)],
        'critic_loss': [[] for _ in range(n_agents)],
        'entropy': [[] for _ in range(n_agents)],
        'episode_return': [[] for _ in range(n_agents)],
        'episode_length': [],
        'success_rate': [],
        'safety_violations': [],
    }
    
    best_return = -float('inf')
    
    for iteration in range(config['max_iterations']):
        # Phase 2: Decentralized data collection
        buffers = collect_rollout(
            env, actors, critics,
            n_steps=config['rollout_length'],
            use_global_state=False
        )
        
        for i in range(n_agents):
            # Phase 3: Independent advantage estimation
            compute_advantages_for_buffer(buffers[i], config['gamma'], config['lam'])
            
            # Phase 4: Independent PPO optimization
            data = buffers[i].to_tensors()
            metrics = optimizers[i].update(data)
            
            # Record training metrics
            history['total_loss'][i].append(metrics['total_loss'])
            history['actor_loss'][i].append(metrics['actor_loss'])
            history['critic_loss'][i].append(metrics['critic_loss'])
            history['entropy'][i].append(metrics['entropy'])
            history['episode_return'][i].append(np.sum(buffers[i].rewards))
        
        # Evaluate current policy
        eval_metrics = evaluate_policy(env, actors, n_episodes=5)
        history['episode_length'].append(eval_metrics['episode_length'])
        history['success_rate'].append(eval_metrics['success_rate'])
        history['safety_violations'].append(eval_metrics['safety_violations'])
        
        # Save best checkpoint
        mean_return = np.mean([history['episode_return'][i][-1] for i in range(n_agents)])
        if mean_return > best_return:
            best_return = mean_return
            save_checkpoint(actors, critics, f'ippo_best_{mode}.pkl')
        
        if verbose and (iteration + 1) % 50 == 0:
            print(f"Iter {iteration+1}/{config['max_iterations']} | "
                  f"Return: {mean_return:.2f} | "
                  f"Success: {eval_metrics['success_rate']:.2%} | "
                  f"Length: {eval_metrics['episode_length']:.1f}")
    
    # Save final checkpoint and training metrics
    save_checkpoint(actors, critics, f'ippo_final_{mode}.pkl')
    np.savez(f'ippo_training_{mode}.npz',
             **{f'{k}_agent{i}': np.array(v) for k, agents in history.items()
                if isinstance(agents, list) and len(agents) == n_agents
                for i, v in enumerate(agents)},
             episode_length=np.array(history['episode_length']),
             success_rate=np.array(history['success_rate']),
             safety_violations=np.array(history['safety_violations']))
    
    return actors, critics, history


def evaluate_policy(env, actors, n_episodes=10, use_agent_id=False):
    """Evaluate current policy over multiple episodes."""
    total_returns = []
    episode_lengths = []
    successes = 0
    violations = 0
    
    for ep in range(n_episodes):
        obs, info = env.reset()
        ep_return = np.zeros(env.N_AGENTS)
        
        for t in range(env.MAX_STEPS):
            actions = []
            for i in range(env.N_AGENTS):
                obs_t = torch.FloatTensor(obs[i]).unsqueeze(0).to(device)
                with torch.no_grad():
                    if use_agent_id:
                        probs = actors[i](obs_t, agent_id=i) if hasattr(actors[i], 'use_agent_id') else actors[i](obs_t)
                    else:
                        probs = actors[i](obs_t)
                    action = probs.argmax(dim=-1).item()  # greedy
                actions.append(action)
            
            obs, rewards, terminated, truncated, info = env.step(actions)
            ep_return += rewards
            
            if terminated:
                successes += 1
                episode_lengths.append(t + 1)
                break
            if truncated:
                violations += 1
                break
        
        total_returns.append(ep_return.mean())
    
    return {
        'avg_return': np.mean(total_returns),
        'episode_length': np.mean(episode_lengths) if episode_lengths else env.MAX_STEPS,
        'success_rate': successes / n_episodes,
        'safety_violations': violations / n_episodes,
    }


def save_checkpoint(actors, critics, filename):
    """Save actor and critic network weights."""
    checkpoint = {
        'actors': [a.state_dict() for a in actors],
        'critics': [c.state_dict() for c in critics],
    }
    with open(filename, 'wb') as f:
        pickle.dump(checkpoint, f)


def load_checkpoint(actors, critics, filename):
    """Load actor and critic network weights."""
    with open(filename, 'rb') as f:
        checkpoint = pickle.load(f)
    for i, a in enumerate(actors):
        a.load_state_dict(checkpoint['actors'][i])
    for i, c in enumerate(critics):
        c.load_state_dict(checkpoint['critics'][i])

In [ ]:
# Train IPPO - Fixed mode
print("=" * 60)
print("Training IPPO - Fixed Dirt Positions")
print("=" * 60)
ippo_actors_fixed, ippo_critics_fixed, ippo_history_fixed = train_ippo(IPPO_CONFIG, mode="fixed")

In [ ]:
# Train IPPO - Random mode
print("=" * 60)
print("Training IPPO - Random Dirt Positions")
print("=" * 60)
ippo_actors_random, ippo_critics_random, ippo_history_random = train_ippo(IPPO_CONFIG, mode="random")

### 3.3 IPPO Training Curves

In [ ]:
def plot_training_curves(history_fixed, history_random, title_prefix="IPPO"):
    """Plot training curves side-by-side for fixed and random modes."""
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    metrics = ['total_loss', 'actor_loss', 'critic_loss', 'entropy']
    titles = ['Total Loss', 'Actor Loss', 'Critic Loss', 'Entropy Bonus']
    
    for col, (metric, title) in enumerate(zip(metrics, titles)):
        for i in range(3):
            axes[0, col].plot(history_fixed[metric][i], label=f'Agent {i+1}', alpha=0.7)
            axes[1, col].plot(history_random[metric][i], label=f'Agent {i+1}', alpha=0.7)
        
        axes[0, col].set_title(f'{title} (Fixed)')
        axes[1, col].set_title(f'{title} (Random)')
        axes[0, col].legend(fontsize=7)
        axes[1, col].legend(fontsize=7)
        axes[1, col].set_xlabel('Iteration')
    
    fig.suptitle(f'{title_prefix} Training Curves', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_training_curves(ippo_history_fixed, ippo_history_random, "IPPO")

### 3.4 IPPO Performance Evaluation

In [ ]:
def plot_performance_metrics(history_fixed, history_random, title_prefix="IPPO"):
    """Plot performance metrics: return, episode length, success rate, safety violations."""
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    
    # Episode return per agent
    for i in range(3):
        axes[0, 0].plot(history_fixed['episode_return'][i], label=f'Agent {i+1}', alpha=0.7)
        axes[1, 0].plot(history_random['episode_return'][i], label=f'Agent {i+1}', alpha=0.7)
    axes[0, 0].set_title('Episode Return (Fixed)')
    axes[1, 0].set_title('Episode Return (Random)')
    
    # Episode length
    axes[0, 1].plot(history_fixed['episode_length'])
    axes[1, 1].plot(history_random['episode_length'])
    axes[0, 1].set_title('Episode Length (Fixed)')
    axes[1, 1].set_title('Episode Length (Random)')
    
    # Success rate
    axes[0, 2].plot(history_fixed['success_rate'])
    axes[1, 2].plot(history_random['success_rate'])
    axes[0, 2].set_title('Success Rate (Fixed)')
    axes[1, 2].set_title('Success Rate (Random)')
    
    # Safety violations
    axes[0, 3].plot(history_fixed['safety_violations'])
    axes[1, 3].plot(history_random['safety_violations'])
    axes[0, 3].set_title('Safety Violations (Fixed)')
    axes[1, 3].set_title('Safety Violations (Random)')
    
    for ax_row in axes:
        for ax in ax_row:
            ax.legend(fontsize=7) if ax.get_legend() else None
            ax.set_xlabel('Iteration')
    
    fig.suptitle(f'{title_prefix} Performance Metrics', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_performance_metrics(ippo_history_fixed, ippo_history_random, "IPPO")

### 3.5 IPPO Visual Demonstration

In [ ]:
def demonstrate_policy(env, actors, title="Policy Demo", use_agent_id=False, n_steps=None):
    """Run a visual demonstration of the learned policy."""
    obs, info = env.reset(seed=123)
    max_steps = n_steps or env.MAX_STEPS
    
    # Show initial and key frames
    frames_to_show = [0, max_steps // 4, max_steps // 2, 3 * max_steps // 4, max_steps - 1]
    fig, axes = plt.subplots(1, min(5, max_steps), figsize=(25, 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    
    frame_idx = 0
    total_rewards = np.zeros(env.N_AGENTS)
    
    for t in range(max_steps):
        if t in frames_to_show and frame_idx < len(axes):
            env.render(ax=axes[frame_idx])
            frame_idx += 1
        
        actions = []
        for i in range(env.N_AGENTS):
            obs_t = torch.FloatTensor(obs[i]).unsqueeze(0).to(device)
            with torch.no_grad():
                if use_agent_id and hasattr(actors[i], 'use_agent_id'):
                    probs = actors[i](obs_t, agent_id=i)
                else:
                    probs = actors[i](obs_t)
                action = probs.argmax(dim=-1).item()
            actions.append(action)
        
        obs, rewards, terminated, truncated, info = env.step(actions)
        total_rewards += rewards
        
        if terminated or truncated:
            if frame_idx < len(axes):
                env.render(ax=axes[frame_idx])
            break
    
    fig.suptitle(f"{title} | Total Rewards: {total_rewards} | Steps: {t+1}", fontsize=12)
    plt.tight_layout()
    plt.show()
    
    return total_rewards, t + 1

# Load best IPPO checkpoint and demonstrate
print("=== IPPO Visual Demo (Fixed Mode) ===")
demo_env = CleaningGridWorld(mode="fixed")
demo_actors = [ActorNetwork(obs_channels=5, hidden_sizes=IPPO_CONFIG['hidden_sizes']).to(device)
               for _ in range(3)]
demo_critics = [CriticNetwork(obs_channels=5, hidden_sizes=IPPO_CONFIG['hidden_sizes']).to(device)
                for _ in range(3)]

try:
    load_checkpoint(demo_actors, demo_critics, 'ippo_best_fixed.pkl')
    demonstrate_policy(demo_env, demo_actors, title="IPPO - Fixed Mode")
except FileNotFoundError:
    print("Checkpoint not found. Run training first.")

---
# Task 4: Multi-Agent Proximal Policy Optimization (MAPPO)

**Deliverables:**
1. Modular codebase with shared actor (one-hot agent ID) and centralized critic
2. Training results: training curves for fixed & random modes
3. Performance evaluation: episode return, length, success rate, safety violations
4. Visual demonstration using best checkpoint

### 4.1 MAPPO Hyperparameters

In [ ]:
MAPPO_CONFIG = {
    'gamma': 0.95,
    'lam': 0.95,
    'epsilon': 0.2,
    'actor_lr': 5e-4,
    'critic_lr': 5e-4,
    'rollout_length': 400,
    'minibatch_size': 200,
    'epochs': 10,
    'max_iterations': 500,
    'hidden_sizes': (128, 64),
    'c1': 0.5,
    'c2': 0.01,
    'weight_decay': 1e-5,
    'seed': 42,
}

print("MAPPO Hyperparameters:")
for k, v in MAPPO_CONFIG.items():
    print(f"  {k}: {v}")

### 4.2 MAPPO Training Loop

In [ ]:
def train_mappo(config, mode="fixed", verbose=True):
    """Train MAPPO: shared actor with agent ID, centralized critic.
    
    Implements Algorithms 5-8.
    Key difference from IPPO:
    - Single shared actor network with one-hot agent ID input
    - Centralized critic that takes global state (all agents' observations)
    """
    env = CleaningGridWorld(mode=mode, seed=config['seed'])
    n_agents = env.N_AGENTS
    
    # Phase 1A: Shared actor with agent ID (one per agent for independent optimization)
    actors = [
        ActorNetwork(
            obs_channels=5, hidden_sizes=config['hidden_sizes'],
            use_agent_id=True, n_agents=n_agents
        ).to(device)
        for _ in range(n_agents)
    ]
    
    # Phase 1B: Centralized critic on global state (12x12x15)
    critics = [
        CriticNetwork(
            obs_channels=5 * n_agents,  # 15 channels = 5 per agent * 3 agents
            hidden_sizes=config['hidden_sizes']
        ).to(device)
        for _ in range(n_agents)
    ]
    
    optimizers = [
        PPOOptimizer(
            actors[i], critics[i],
            actor_lr=config['actor_lr'],
            critic_lr=config['critic_lr'],
            epsilon=config['epsilon'],
            c1=config['c1'],
            c2=config['c2'],
            epochs=config['epochs'],
            minibatch_size=config['minibatch_size'],
            weight_decay=config['weight_decay'],
            use_agent_id=True,
            agent_id=i,
        )
        for i in range(n_agents)
    ]
    
    history = {
        'total_loss': [[] for _ in range(n_agents)],
        'actor_loss': [[] for _ in range(n_agents)],
        'critic_loss': [[] for _ in range(n_agents)],
        'entropy': [[] for _ in range(n_agents)],
        'episode_return': [[] for _ in range(n_agents)],
        'episode_length': [],
        'success_rate': [],
        'safety_violations': [],
    }
    
    best_return = -float('inf')
    
    for iteration in range(config['max_iterations']):
        # Phase 2: Data collection with global state (Algorithm 6)
        buffers = collect_rollout(
            env, actors, critics,
            n_steps=config['rollout_length'],
            use_global_state=True  # key difference: store global state
        )
        
        for i in range(n_agents):
            # Phase 3: Centralized advantage estimation (Algorithm 7)
            compute_advantages_for_buffer(buffers[i], config['gamma'], config['lam'])
            
            # Phase 4: Centralized PPO optimization (Algorithm 8)
            data = buffers[i].to_tensors()
            metrics = optimizers[i].update(data)
            
            history['total_loss'][i].append(metrics['total_loss'])
            history['actor_loss'][i].append(metrics['actor_loss'])
            history['critic_loss'][i].append(metrics['critic_loss'])
            history['entropy'][i].append(metrics['entropy'])
            history['episode_return'][i].append(np.sum(buffers[i].rewards))
        
        # Evaluate
        eval_metrics = evaluate_policy(env, actors, n_episodes=5, use_agent_id=True)
        history['episode_length'].append(eval_metrics['episode_length'])
        history['success_rate'].append(eval_metrics['success_rate'])
        history['safety_violations'].append(eval_metrics['safety_violations'])
        
        mean_return = np.mean([history['episode_return'][i][-1] for i in range(n_agents)])
        if mean_return > best_return:
            best_return = mean_return
            save_checkpoint(actors, critics, f'mappo_best_{mode}.pkl')
        
        if verbose and (iteration + 1) % 50 == 0:
            print(f"Iter {iteration+1}/{config['max_iterations']} | "
                  f"Return: {mean_return:.2f} | "
                  f"Success: {eval_metrics['success_rate']:.2%} | "
                  f"Length: {eval_metrics['episode_length']:.1f}")
    
    save_checkpoint(actors, critics, f'mappo_final_{mode}.pkl')
    np.savez(f'mappo_training_{mode}.npz',
             **{f'{k}_agent{i}': np.array(v) for k, agents in history.items()
                if isinstance(agents, list) and len(agents) == n_agents
                for i, v in enumerate(agents)},
             episode_length=np.array(history['episode_length']),
             success_rate=np.array(history['success_rate']),
             safety_violations=np.array(history['safety_violations']))
    
    return actors, critics, history

In [ ]:
# Train MAPPO - Fixed mode
print("=" * 60)
print("Training MAPPO - Fixed Dirt Positions")
print("=" * 60)
mappo_actors_fixed, mappo_critics_fixed, mappo_history_fixed = train_mappo(MAPPO_CONFIG, mode="fixed")

In [ ]:
# Train MAPPO - Random mode
print("=" * 60)
print("Training MAPPO - Random Dirt Positions")
print("=" * 60)
mappo_actors_random, mappo_critics_random, mappo_history_random = train_mappo(MAPPO_CONFIG, mode="random")

### 4.3 MAPPO Training Curves

In [ ]:
plot_training_curves(mappo_history_fixed, mappo_history_random, "MAPPO")

### 4.4 MAPPO Performance Evaluation

In [ ]:
plot_performance_metrics(mappo_history_fixed, mappo_history_random, "MAPPO")

### 4.5 MAPPO Visual Demonstration

In [ ]:
print("=== MAPPO Visual Demo (Fixed Mode) ===")
demo_env_mappo = CleaningGridWorld(mode="fixed")
demo_actors_mappo = [
    ActorNetwork(obs_channels=5, hidden_sizes=MAPPO_CONFIG['hidden_sizes'],
                 use_agent_id=True, n_agents=3).to(device)
    for _ in range(3)
]
demo_critics_mappo = [
    CriticNetwork(obs_channels=15, hidden_sizes=MAPPO_CONFIG['hidden_sizes']).to(device)
    for _ in range(3)
]

try:
    load_checkpoint(demo_actors_mappo, demo_critics_mappo, 'mappo_best_fixed.pkl')
    demonstrate_policy(demo_env_mappo, demo_actors_mappo, title="MAPPO - Fixed Mode", use_agent_id=True)
except FileNotFoundError:
    print("Checkpoint not found. Run training first.")

---
# Task 5: Comprehensive Evaluation

Compare IPPO and MAPPO across 4 evaluation aspects:
1. **Learning Performance** - success rate, rewards over 400 test episodes (box plots)
2. **Task Efficiency & Coordination** - Global RCR, RCR heatmap
3. **Agents' Contribution** - advantage estimates per agent
4. **Behavioural Diversity** - JSD matrix between agent policies

### 5.1 Learning Performance (400 test episodes)

In [ ]:
def evaluate_comprehensive(env, actors, n_episodes=400, use_agent_id=False, seed=999):
    """Run comprehensive evaluation over many episodes.
    
    Returns per-agent returns, success rates, episode lengths, violations,
    visit/revisit maps, and agent advantages.
    """
    returns_per_agent = [[] for _ in range(env.N_AGENTS)]
    episode_lengths = []
    successes = 0
    violations = 0
    
    # Aggregated maps
    agg_visit_map = np.zeros((env.GRID_SIZE, env.GRID_SIZE))
    agg_revisit_map = np.zeros((env.GRID_SIZE, env.GRID_SIZE))
    
    # Agent advantages for contribution analysis
    agent_advantages = [[] for _ in range(env.N_AGENTS)]
    
    # Policy distributions for JSD
    policy_distributions = [[] for _ in range(env.N_AGENTS)]
    
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        ep_return = np.zeros(env.N_AGENTS)
        
        for t in range(env.MAX_STEPS):
            actions = []
            for i in range(env.N_AGENTS):
                obs_t = torch.FloatTensor(obs[i]).unsqueeze(0).to(device)
                with torch.no_grad():
                    if use_agent_id:
                        probs = actors[i](obs_t, agent_id=i)
                    else:
                        probs = actors[i](obs_t)
                    action = probs.argmax(dim=-1).item()
                    policy_distributions[i].append(probs.cpu().numpy().flatten())
                actions.append(action)
            
            obs, rewards, terminated, truncated, info = env.step(actions)
            ep_return += rewards
            
            # Advantage = r + gamma*V(s') - V(s)
            # Simplified: just use reward as proxy during evaluation
            for i in range(env.N_AGENTS):
                agent_advantages[i].append(rewards[i])
            
            if terminated:
                successes += 1
                episode_lengths.append(t + 1)
                break
            if truncated:
                violations += 1
                break
        
        for i in range(env.N_AGENTS):
            returns_per_agent[i].append(ep_return[i])
        
        agg_visit_map += env.visit_map
        agg_revisit_map += env.revisit_map
    
    return {
        'returns': returns_per_agent,
        'episode_lengths': episode_lengths,
        'success_rate': successes / n_episodes,
        'violation_rate': violations / n_episodes,
        'visit_map': agg_visit_map,
        'revisit_map': agg_revisit_map,
        'advantages': agent_advantages,
        'policy_distributions': policy_distributions,
    }

# Run evaluation for all 4 settings
print("Evaluating IPPO (fixed)...")
# eval_ippo_fixed = evaluate_comprehensive(env_fixed, ippo_actors_fixed, use_agent_id=False)
# print("Evaluating IPPO (random)...")
# eval_ippo_random = evaluate_comprehensive(env_random, ippo_actors_random, use_agent_id=False)
# print("Evaluating MAPPO (fixed)...")
# eval_mappo_fixed = evaluate_comprehensive(env_fixed, mappo_actors_fixed, use_agent_id=True)
# print("Evaluating MAPPO (random)...")
# eval_mappo_random = evaluate_comprehensive(env_random, mappo_actors_random, use_agent_id=True)

# TODO: Uncomment above after training is complete
print("(Uncomment evaluation calls after training)")

In [ ]:
def plot_box_comparison(eval_results, labels, title="Learning Performance Comparison"):
    """Generate box plots comparing success rate and returns across modes."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Box plot of returns per agent
    all_returns = []
    box_labels = []
    for eval_res, label in zip(eval_results, labels):
        for i in range(3):
            all_returns.append(eval_res['returns'][i])
            box_labels.append(f"{label}\nAgent {i+1}")
    
    axes[0].boxplot(all_returns, labels=box_labels)
    axes[0].set_title('Return Distribution per Agent per Mode')
    axes[0].set_ylabel('Episode Return')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Success rate and violation rate bar chart
    x = np.arange(len(labels))
    success = [e['success_rate'] for e in eval_results]
    violations = [e['violation_rate'] for e in eval_results]
    
    axes[1].bar(x - 0.15, success, 0.3, label='Success Rate', color='green', alpha=0.7)
    axes[1].bar(x + 0.15, violations, 0.3, label='Violation Rate', color='red', alpha=0.7)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels)
    axes[1].set_title('Success & Violation Rates')
    axes[1].legend()
    axes[1].set_ylabel('Rate')
    
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# TODO: Uncomment after evaluation
# plot_box_comparison(
#     [eval_ippo_fixed, eval_ippo_random, eval_mappo_fixed, eval_mappo_random],
#     ['IPPO Fixed', 'IPPO Random', 'MAPPO Fixed', 'MAPPO Random']
# )
print("Box plot function ready. Uncomment after training/evaluation.")

### 5.2 Task Efficiency & Coordination (Global RCR, RCR Heatmap)

In [ ]:
def plot_rcr_analysis(eval_results, labels):
    """Plot Global RCR values and RCR heatmaps."""
    n = len(eval_results)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    
    global_rcrs = []
    
    for idx, (eval_res, label) in enumerate(zip(eval_results, labels)):
        visit_map = eval_res['visit_map']
        revisit_map = eval_res['revisit_map']
        
        # Global RCR
        total_visits = visit_map.sum()
        total_revisits = revisit_map.sum()
        global_rcr = total_revisits / max(1, total_visits)
        global_rcrs.append(global_rcr)
        
        # Per-cell RCR map
        rcr_map = np.zeros_like(visit_map, dtype=float)
        mask = visit_map > 0
        rcr_map[mask] = revisit_map[mask] / visit_map[mask]
        
        im = axes[idx].imshow(rcr_map, cmap='YlOrRd', vmin=0, vmax=1)
        axes[idx].set_title(f"{label}\nGlobal RCR: {global_rcr:.4f}")
        axes[idx].set_xticks(range(12))
        axes[idx].set_yticks(range(12))
        axes[idx].set_xticklabels(range(1, 13))
        axes[idx].set_yticklabels(range(1, 13))
        plt.colorbar(im, ax=axes[idx], shrink=0.8)
    
    fig.suptitle('RCR Heatmaps (lower = better coordination)', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return global_rcrs

# TODO: Uncomment after evaluation
# rcrs = plot_rcr_analysis(
#     [eval_ippo_fixed, eval_ippo_random, eval_mappo_fixed, eval_mappo_random],
#     ['IPPO Fixed', 'IPPO Random', 'MAPPO Fixed', 'MAPPO Random']
# )
print("RCR analysis function ready.")

### 5.3 Agents' Contribution (Advantage Analysis)

In [ ]:
def plot_agent_contribution(eval_results, labels):
    """Plot per-agent advantage estimates to measure contribution."""
    fig, axes = plt.subplots(1, len(eval_results), figsize=(6 * len(eval_results), 5))
    if len(eval_results) == 1:
        axes = [axes]
    
    for idx, (eval_res, label) in enumerate(zip(eval_results, labels)):
        agent_means = []
        agent_stds = []
        for i in range(3):
            adv = np.array(eval_res['advantages'][i])
            agent_means.append(adv.mean())
            agent_stds.append(adv.std())
        
        x = np.arange(3)
        axes[idx].bar(x, agent_means, yerr=agent_stds, capsize=5,
                      color=['gold', 'green', 'royalblue'], alpha=0.7)
        axes[idx].set_xticks(x)
        axes[idx].set_xticklabels(['Agent 1', 'Agent 2', 'Agent 3'])
        axes[idx].set_title(f"{label}")
        axes[idx].set_ylabel('Mean Advantage (Contribution)')
    
    fig.suptitle('Agent Contribution Analysis', fontsize=14)
    plt.tight_layout()
    plt.show()

# TODO: Uncomment after evaluation
# plot_agent_contribution(
#     [eval_ippo_fixed, eval_ippo_random, eval_mappo_fixed, eval_mappo_random],
#     ['IPPO Fixed', 'IPPO Random', 'MAPPO Fixed', 'MAPPO Random']
# )
print("Agent contribution function ready.")

### 5.4 Behavioural Diversity (JSD Matrix)

In [ ]:
def compute_jsd(p, q):
    """Compute Jensen-Shannon Divergence between two distributions."""
    p = np.array(p, dtype=np.float64)
    q = np.array(q, dtype=np.float64)
    # Clip to avoid log(0)
    p = np.clip(p, 1e-10, 1.0)
    q = np.clip(q, 1e-10, 1.0)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return 0.5 * kl_pm + 0.5 * kl_qm


def plot_jsd_matrix(eval_results, labels):
    """Plot pairwise JSD divergence matrices for 3 agents."""
    fig, axes = plt.subplots(1, len(eval_results), figsize=(6 * len(eval_results), 5))
    if len(eval_results) == 1:
        axes = [axes]
    
    for idx, (eval_res, label) in enumerate(zip(eval_results, labels)):
        # Average policy distribution per agent
        avg_policies = []
        for i in range(3):
            dists = np.array(eval_res['policy_distributions'][i])
            avg_policies.append(dists.mean(axis=0))
        
        # Compute pairwise JSD
        jsd_matrix = np.zeros((3, 3))
        for i in range(3):
            for j in range(3):
                if i != j:
                    jsd_matrix[i, j] = compute_jsd(avg_policies[i], avg_policies[j])
        
        im = axes[idx].imshow(jsd_matrix, cmap='Blues', vmin=0)
        axes[idx].set_xticks(range(3))
        axes[idx].set_yticks(range(3))
        axes[idx].set_xticklabels(['Agent 1', 'Agent 2', 'Agent 3'])
        axes[idx].set_yticklabels(['Agent 1', 'Agent 2', 'Agent 3'])
        axes[idx].set_title(f"{label}")
        
        # Annotate values
        for i in range(3):
            for j in range(3):
                axes[idx].text(j, i, f"{jsd_matrix[i, j]:.4f}",
                             ha='center', va='center', fontsize=10)
        
        plt.colorbar(im, ax=axes[idx], shrink=0.8)
    
    fig.suptitle('Jensen-Shannon Divergence Between Agent Policies', fontsize=14)
    plt.tight_layout()
    plt.show()

# TODO: Uncomment after evaluation
# plot_jsd_matrix(
#     [eval_ippo_fixed, eval_ippo_random, eval_mappo_fixed, eval_mappo_random],
#     ['IPPO Fixed', 'IPPO Random', 'MAPPO Fixed', 'MAPPO Random']
# )
print("JSD matrix function ready.")

### 5.5 Summary Comparison Table

In [ ]:
def print_summary_table(eval_results, labels):
    """Print a summary comparison table of all metrics."""
    print(f"{'Metric':<30}", end="")
    for label in labels:
        print(f"{label:<20}", end="")
    print()
    print("-" * (30 + 20 * len(labels)))
    
    # Average return
    print(f"{'Avg Return':<30}", end="")
    for e in eval_results:
        avg = np.mean([np.mean(e['returns'][i]) for i in range(3)])
        print(f"{avg:<20.4f}", end="")
    print()
    
    # Episode length
    print(f"{'Avg Episode Length':<30}", end="")
    for e in eval_results:
        if e['episode_lengths']:
            print(f"{np.mean(e['episode_lengths']):<20.2f}", end="")
        else:
            print(f"{'N/A':<20}", end="")
    print()
    
    # Success rate
    print(f"{'Success Rate':<30}", end="")
    for e in eval_results:
        print(f"{e['success_rate']:<20.4f}", end="")
    print()
    
    # Violation rate
    print(f"{'Safety Violation Rate':<30}", end="")
    for e in eval_results:
        print(f"{e['violation_rate']:<20.4f}", end="")
    print()
    
    # Global RCR
    print(f"{'Global RCR':<30}", end="")
    for e in eval_results:
        rcr = e['revisit_map'].sum() / max(1, e['visit_map'].sum())
        print(f"{rcr:<20.4f}", end="")
    print()

# TODO: Uncomment after evaluation
# print_summary_table(
#     [eval_ippo_fixed, eval_ippo_random, eval_mappo_fixed, eval_mappo_random],
#     ['IPPO Fixed', 'IPPO Random', 'MAPPO Fixed', 'MAPPO Random']
# )
print("Summary table function ready.")

---
## End of Notebook

**Submission checklist:**
- [ ] Task 1: Environment with visualization, random demo, collision verification
- [ ] Task 2: Actor/Critic, Rollout buffer, GAE, Optimizer - each with tests
- [ ] Task 3: IPPO training (fixed + random), curves, metrics, demo
- [ ] Task 4: MAPPO training (fixed + random), curves, metrics, demo
- [ ] Task 5: Box plots, RCR heatmaps, contribution analysis, JSD matrices
- [ ] All figures rendered and visible
- [ ] Saved artefacts: checkpoints (.pkl), training metrics (.npz)